# 다이캐스팅 공정 데이터 EDA 파이프라인
> **구성**: 데이터 로드 → 구조 파악 → 중복 확인 → 결측값 처리 → 이상치 처리 → 타겟/파생변수 생성 → 분포 시각화 → 불량 분포 시각화

## STEP 0. 라이브러리 임포트 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── 데이터 로드 (멀티헤더)
df_raw = pd.read_csv(
    'DieCasting_Quality_Raw_Data.csv',
    header=[0, 1]
)
print(f"원본 Shape: {df_raw.shape}")
df_raw.head(5)

## STEP 1. 데이터 구조 파악

In [ ]:
# 컬럼 목록 (멀티헤더 상태)
print("컬럼 그룹:", df_raw.columns.get_level_values(0).unique().tolist())
print("전체 컬럼 수:", len(df_raw.columns))
df_raw.info()

In [ ]:
# 기초 통계량 (멀티헤더 상태)
df_raw.describe()

In [ ]:
# 고유값 수 확인
df_raw.nunique()

## STEP 2. 중복 데이터 확인
> id는 고유하지만, id를 제외한 실질 데이터 기준으로 중복 여부 확인

In [ ]:
# 전체 중복 (id 포함)
print("전체 중복 행:", df_raw.duplicated().sum())

# id 제외 실질 중복
df_dropid = df_raw.drop(columns=[('Process', 'id')]).copy()
print("id 제외 중복 행:", df_dropid.duplicated().sum())

In [ ]:
# 중복 행 상세 확인
duplicate_rows = df_dropid[df_dropid.duplicated(keep=False)]
print(f"중복 관련 행 수: {len(duplicate_rows)}")
duplicate_rows.sort_values(by=('Process', 'Shot')).head(20)

In [ ]:
# 중복 행 중 불량 vs 양품 분리 확인
dup_defects = duplicate_rows[duplicate_rows['Defects'].sum(axis=1) > 0]
dup_good    = duplicate_rows[duplicate_rows['Defects'].sum(axis=1) == 0]
print(f"중복 중 불량: {len(dup_defects)}건 / 양품: {len(dup_good)}건")

In [ ]:
# Shot 번호 고유값 여부 확인 (Product_Type 내 순번이므로 전체에서 중복 발생)
shot_cnt = df_raw['Process']['Shot'].value_counts()
print(f"Shot 고유값 수: {df_raw['Process']['Shot'].nunique()}")
print(f"전체 행 수: {len(df_raw)}")
print("→ Shot은 Product_Type 내 순번으로 전체에서 중복됨")
shot_cnt.head(10)

## STEP 3. 헤더 정리 및 수치형 변환
> 멀티헤더 → 단일헤더, 전체 수치형 변환 (이후 모든 작업은 df 기준)

In [ ]:
df = df_raw.copy()

# 멀티헤더 → 단일헤더 (2행 컬럼명 사용)
df.columns = df.columns.get_level_values(1).str.strip()

# id 제외 전체 컬럼 수치형 변환
df.iloc[:, 1:] = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

print(f"✅ 변환 완료: {df.shape[0]}행 × {df.shape[1]}열")
print(f"   컬럼 목록: {df.columns.tolist()}")

## STEP 4. 결측값 확인 및 처리

In [ ]:
# 결측값 현황
missing_counts = df.isnull().sum()
missing_cols   = missing_counts[missing_counts > 0]

missing_df = pd.DataFrame({
    'missing_count': missing_cols,
    'missing_ratio(%)': df[missing_cols.index].isnull().mean() * 100
}).sort_values('missing_count', ascending=False)

print("결측값 보유 컬럼:")
display(missing_df)

In [ ]:
# Factory 센서 6개 컬럼 → Product_Type별 중앙값으로 대체
factory_cols = missing_cols.index

df[factory_cols] = (
    df.groupby('Product_Type')[factory_cols]
      .transform(lambda x: x.fillna(x.median()))
)

print(f"✅ 결측값 처리 완료 → 잔여 결측값: {df.isnull().sum().sum()}건")
print("   처리 방법: Product_Type 그룹별 중앙값 대체")

## STEP 5. 이상치 탐지 및 처리 (IQR 기준)
> Process 변수와 Sensor 변수 각각 탐지 후 IQR Capping 적용

In [ ]:
# 이상치 탐지 함수 (팀원 EDA 재사용)
def detect_outliers_iqr(dataframe, cols):
    outlier_info = {}
    for col in cols:
        Q1, Q3 = dataframe[col].quantile([0.25, 0.75])
        IQR    = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        mask   = (dataframe[col] < lower) | (dataframe[col] > upper)
        outlier_info[col] = {
            'count': mask.sum(),
            'pct(%)': f"{mask.mean() * 100:.2f}",
            'lower_bound': round(lower, 3),
            'upper_bound': round(upper, 3)
        }
    result = pd.DataFrame(outlier_info).T.sort_values('count', ascending=False)
    return result[result['count'] > 0]

In [ ]:
# Process 변수 이상치 확인
process_num_cols = [
    'Velocity_1', 'Velocity_2', 'Velocity_3', 'High_Velocity',
    'Cylinder_Pressure', 'Rapid_Rise_Time', 'Biscuit_Thickness',
    'Clamping_Force', 'Cycle_Time', 'Pressure_Rise_Time',
    'Casting_Pressure', 'Spray_Time', 'Spray_1_Time', 'Spray_2_Time'
]

print("[ Process 변수 이상치 ]")
display(detect_outliers_iqr(df, process_num_cols))

In [ ]:
# Sensor 변수 이상치 확인
sensor_cols = [
    'Melting_Furnace_Temp', 'Air_Pressure',
    'Coolant_Temp', 'Coolant_Pressure',
    'Factory_Temp', 'Factory_Humidity'
]

print("[ Sensor 변수 이상치 ]")
display(detect_outliers_iqr(df, sensor_cols))

In [ ]:
# IQR Capping 적용 (Process 변수만 — Sensor는 이상치 없음)
outlier_report = []
for col in process_num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    cnt = ((df[col] < lower) | (df[col] > upper)).sum()
    if cnt > 0:
        outlier_report.append({'변수': col, '이상치 수': cnt})
    df[col] = df[col].clip(lower, upper)

print(" IQR Capping 완료 (처리 변수):")
for r in outlier_report:
    print(f"   - {r['변수']}: {r['이상치 수']}건")

## STEP 6. 타겟 변수 및 파생변수 생성

In [ ]:
# ── 타겟 변수 1: 이진 분류 (양품=0 / 불량=1)
defect_cols_all = [
    c for c in df.columns if any(c.startswith(d) for d in [
        'Short_Shot', 'Bubble', 'Exfoliation', 'Blow_Hole',
        'Stain', 'Dent', 'Deformation', 'Contamination',
        'Impurity', 'Crack', 'Scratch', 'Buring_Mark', 'Inclusions'
    ])
]
df['Defect_Status'] = (df[defect_cols_all].sum(axis=1) > 0).astype(int)

# ── 타겟 변수 2: 다중 분류 (불량 유형 레이블)
defect_types = {
    'Short_Shot':  ['Short_Shot_1',  'Short_Shot_2'],
    'Blow_Hole':   ['Blow_Hole_1',   'Blow_Hole_2'],
    'Exfoliation': ['Exfoliation_1', 'Exfoliation_2'],
    'Stain':       ['Stain_1',       'Stain_2'],
    'Deformation': ['Deformation_1', 'Deformation_2'],
    'Bubble':      ['Bubble_1',      'Bubble_2'],
    'Other': [
        'Dent_1', 'Dent_2', 'Contamination_1', 'Contamination_2',
        'Impurity_1', 'Impurity_2', 'Crack_1', 'Crack_2',
        'Scratch_1', 'Scratch_2', 'Buring_Mark_1', 'Buring_Mark_2',
        'Inclusions_1', 'Inclusions_2'
    ]
}

def get_defect_label(row):
    if row['Defect_Status'] == 0:
        return 'Normal'
    for label, cols in defect_types.items():
        valid_cols = [c for c in cols if c in row.index]
        if row[valid_cols].sum() > 0:
            return label
    return 'Other'

df['Defect_Type'] = df.apply(get_defect_label, axis=1)

print(" 타겟 변수 생성 완료")
print(f"\n[Defect_Status 분포]")
print(df['Defect_Status'].value_counts().to_string())
print(f"\n[Defect_Type 분포]")
print(df['Defect_Type'].value_counts().to_string())

In [ ]:
# ── 파생변수 생성
df['Velocity_Avg']       = df[['Velocity_1', 'Velocity_2', 'Velocity_3']].mean(axis=1)
df['Pressure_Diff']      = df['Casting_Pressure'] - df['Cylinder_Pressure']
df['Coolant_Temp_Range'] = df['Coolant_Temp_Max'] - df['Coolant_Temp_Min']

print("✅ 파생변수 생성 완료")
print("   └ Velocity_Avg      : Velocity 1~3 평균")
print("   └ Pressure_Diff     : Casting_Pressure - Cylinder_Pressure")
print("   └ Coolant_Temp_Range: Coolant_Temp_Max - Coolant_Temp_Min")

## STEP 7. 변수별 분포 시각화
### 7-1. Process 변수 히스토그램 + 박스플롯

In [ ]:
cols    = process_num_cols
n_cols  = 4
n_rows  = (len(cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue', alpha=0.7)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Process Variables — Histogram', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.boxplot(y=df[col], ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontsize=11)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Process Variables — Boxplot', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 7-2. Sensor 변수 히스토그램 + 박스플롯

In [ ]:
sensor_vis_cols = [
    'Melting_Furnace_Temp', 'Air_Pressure',
    'Coolant_Temp', 'Coolant_Pressure',
    'Factory_Temp', 'Factory_Humidity'
]
n_cols_s = 3
n_rows_s = (len(sensor_vis_cols) + n_cols_s - 1) // n_cols_s

fig, axes = plt.subplots(n_rows_s, n_cols_s, figsize=(18, 4 * n_rows_s))
axes = axes.flatten()

for i, col in enumerate(sensor_vis_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='darkorange', alpha=0.7)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Sensor Variables — Histogram', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(n_rows_s, n_cols_s, figsize=(18, 4 * n_rows_s))
axes = axes.flatten()

for i, col in enumerate(sensor_vis_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='darkorange')
    axes[i].set_title(col, fontsize=11)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Sensor Variables — Boxplot', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## STEP 8. 불량 데이터 분포 시각화

In [ ]:
# 불량 유형별 발생 건수 바 차트 (팀원 EDA)
df_defects = df[defect_cols_all].copy()
defect_counts = df_defects.sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(16, 5))
defect_counts.plot(kind='bar', color='steelblue', ax=ax, alpha=0.8)
ax.axhline(y=10, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='기준선 (10건)')
ax.set_ylabel('Count')
ax.set_title('Defect Type Occurrence Count (All Columns)', fontsize=13, fontweight='bold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 양품 vs 불량 비율 파이차트 + Defect_Type 막대 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 파이 차트
status_counts = df['Defect_Status'].value_counts()
axes[0].pie(
    status_counts,
    labels=['Normal', 'Defect'],
    autopct='%1.1f%%',
    colors=['#2471A3', '#E74C3C'],
    startangle=90
)
axes[0].set_title('Defect Status (Normal vs Defect)', fontsize=12, fontweight='bold')

# 불량 유형 막대 차트
type_counts = df['Defect_Type'].value_counts()
type_counts.plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8)
axes[1].set_title('Defect Type Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f"\n불량률: {df['Defect_Status'].mean()*100:.1f}%")
print(df['Defect_Type'].value_counts().to_string())

In [ ]:
# Product_Type별 불량률 비교
pt_defect = df.groupby('Product_Type')['Defect_Status'].agg(['sum','count'])
pt_defect['defect_rate(%)'] = (pt_defect['sum'] / pt_defect['count'] * 100).round(2)
pt_defect.columns = ['불량_수', '전체_수', '불량률(%)']
print("Product_Type별 불량 현황:")
display(pt_defect)

## STEP 9. 최종 정제 데이터셋 저장

In [ ]:
feature_cols = (
    ['id', 'Product_Type', 'Shot']
    + process_num_cols
    + ['Melting_Furnace_Temp', 'Air_Pressure', 'Coolant_Temp',
       'Coolant_Pressure', 'Factory_Temp', 'Factory_Humidity']
    + ['Velocity_Avg', 'Pressure_Diff', 'Coolant_Temp_Range']
    + ['Defect_Status', 'Defect_Type']
)

df_clean = df[feature_cols].copy()
df_clean.to_csv('DieCasting_Preprocessed.csv', index=False)

print(f" 저장 완료: DieCasting_Preprocessed.csv")
print(f"   최종 Shape: {df_clean.shape[0]}행 × {df_clean.shape[1]}열")
print(f"   컬럼: {df_clean.columns.tolist()}")
df_clean.head(5)